###Transform Races Data
1. Read Bronze races table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake case 
4. Rename columns to make more meanimgful
5. Remove duplicate records
6. Transform values of column race_name to title case
7. Write the transformed data to races table

Below changes are required to implement incremental load

1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passed in (i.e., filter reading from bronze using the batch_id)
3. Add created_timestamp, updated_timestamp and batch_id to the silver table.
4. Merge the processed data to the silver table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/03.silver_helpers

In [0]:
from pyspark.sql import functions as f

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"


In [0]:
races_df = spark.table(bronze_table).filter(f.col("batch_id") == v_batch_id)

In [0]:
races_selected_df = (
    races_df.select(
        f.col("season"),
        f.col("round"),
        f.col("raceName"),
        f.col("date"),
        f.col("circuitId"),
        f.col("current_timestamp"),
        f.col("source_file"),
        f.col("batch_id")
    )
)

In [0]:
races_renamed_df = (
    races_selected_df.withColumnsRenamed({
        "raceName": "race_name",
        "circuitId": "circuit_id",
        "date": "race_date"
    })
)

In [0]:
#display(races_renamed_df)

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

In [0]:
#display(races_distinct_df)

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn("race_name", f.initcap(f.col("race_name")))
)

In [0]:
#display(races_final_df)

In [0]:
write_to_silver(
    races_final_df,
    silver_table,
    "t.season = s.season AND t.round = s.round",
    [
        "race_name",
        "race_date",
        "circuit_id",
        "current_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
#display(spark.table(silver_table))

In [0]:
%sql
--select * from formula1.silver.races